# CRTS-250 Event Analysis

Builds a hit-level pandas DataFrame from a SPECT+TIMING `.dat` file.

**Format:** one row per hit; events are groupable via `event_number`.

**Cuts applied:**
- Each hit must have HG ADC > 250 (`HG_MIN`)
- The event must have at least 2 such hits on at least 2 distinct layers

**Memory:** the file is never fully loaded — events are read and discarded in chunks of `CHUNK_SIZE`.

Cell1:

In [1]:
import struct
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt
import os
import sys

# Define the unpackers and other helper routines

Cell 2:

In [2]:
# ---------------------------------------------------------------------------
# Binary format constants  (mirror spect_t_unpacker.py)
# ---------------------------------------------------------------------------
_U16 = struct.Struct('<H')
_U64 = struct.Struct('<Q')
_F32 = struct.Struct('<f')
_F64 = struct.Struct('<d')

FILE_HEADER_SIZE = 25  # bytes


def _parse_file_header(f):
    """Read the 25-byte file header; return (time_unit, toa_lsb_ns)."""
    hdr = f.read(FILE_HEADER_SIZE)
    if len(hdr) < FILE_HEADER_SIZE:
        raise ValueError("File too short to contain a valid header")
    time_unit  = hdr[12]
    toa_lsb_ns = _F32.unpack_from(hdr, 13)[0]
    return time_unit, toa_lsb_ns


def _parse_hits(data, time_unit, toa_lsb_ns):
    """
    Parse all hit records from a single event's raw bytes.
    `data` must contain the full event starting at offset 0
    (i.e. including the 27-byte event header).
    Returns a list of dicts.
    """
    size         = _U16.unpack_from(data, 0)[0]
    channel_mask = _U64.unpack_from(data, 19)[0]

    hits = []
    ho   = 27       # hit payload starts after the 27-byte event header
    mask = channel_mask

    while mask and ho + 2 <= size:
        mask   ^= mask & (-mask)            # clear lowest set bit
        channel = data[ho]
        dtype   = data[ho + 1]
        ho     += 2

        energy_lg = energy_hg = toa = tot = None

        if dtype & 0x01:
            energy_lg = _U16.unpack_from(data, ho)[0];  ho += 2
        if dtype & 0x02:
            energy_hg = _U16.unpack_from(data, ho)[0];  ho += 2
        if dtype & 0x10:
            if time_unit == 1:
                toa = int(_F32.unpack_from(data, ho)[0]);  ho += 4
            else:
                toa = _U16.unpack_from(data, ho)[0];       ho += 2
        if dtype & 0x20:
            if time_unit == 1:
                tot = int(_F32.unpack_from(data, ho)[0] / toa_lsb_ns);  ho += 4
            else:
                tot = _U16.unpack_from(data, ho)[0];                     ho += 2

        hits.append(dict(
            channel=channel,
            energy_lg=energy_lg,
            energy_hg=energy_hg,
            toa=toa,
            tot=tot,
        ))

    return hits


def iter_events_chunked(filename, chunk_size=1000):
    """
    Generator — reads `filename` event-by-event without loading the full
    file into memory.  Yields lists of raw-event dicts, each list containing
    at most `chunk_size` events.

    Each yielded dict has keys:
        event_number, board_id, timestamp_us, trigger_id, hits
    where `hits` is a list of per-channel dicts.
    """
    with open(filename, 'rb') as f:
        time_unit, toa_lsb_ns = _parse_file_header(f)
        print(f"Header parsed — time_unit={'ns' if time_unit else 'LSB'}, "
              f"toa_lsb={toa_lsb_ns:.3f} ns")

        chunk      = []
        event_num  = 0

        while True:
            # Read event size field (2 bytes) — this is our EOF sentinel
            size_buf = f.read(2)
            if len(size_buf) < 2:
                break                       # clean EOF

            size = _U16.unpack(size_buf)[0]
            if size < 27:
                break                       # corrupt / truncated

            rest = f.read(size - 2)
            if len(rest) < size - 2:
                break                       # truncated event at end of file

            raw   = size_buf + rest
            board = raw[2]
            ts_us = _F64.unpack_from(raw, 3)[0]
            trig  = _U64.unpack_from(raw, 11)[0]
            hits  = _parse_hits(raw, time_unit, toa_lsb_ns)

            chunk.append(dict(
                event_number = event_num,
                board_id     = board,
                timestamp_us = ts_us,
                trigger_id   = trig,
                hits         = hits,
            ))
            event_num += 1

            if len(chunk) >= chunk_size:
                yield chunk
                chunk = []

        if chunk:              # flush the final partial chunk
            yield chunk

# Load the mapping

In [3]:
# ---------------------------------------------------------------------------
# Channel mapping  (channel -> (layer, position))
# ---------------------------------------------------------------------------

def load_mapping(filepath):
    """
    Parse a mapping file (any number of rows, any number of channels per row).
    Returns dict: channel_number -> (layer, position)
    where layer = row index, position = column index within that row.
    """
    mapping = {}
    with open(filepath) as fh:
        for layer, line in enumerate(fh):
            for position, ch in enumerate(int(x) for x in line.strip().split(',')):
                mapping[ch] = (layer, position)
    return mapping


board0_map = load_mapping('CRTS_25_mapping_board0.txt')   # side = 0
board1_map = load_mapping('CRTS_25_mapping_board1.txt')   # side = 1
board2_map = load_mapping('CTS_mapping_board2.txt')       # side = 2  (strip: 2 rows × 4 ch)

print(f"Board 0 mapping: {len(board0_map)} channels")
print(f"Board 1 mapping: {len(board1_map)} channels")
print(f"Board 2 mapping: {len(board2_map)} channels  →  {board2_map}")

Board 0 mapping: 64 channels
Board 1 mapping: 64 channels
Board 2 mapping: 8 channels  →  {1: (0, 0), 3: (0, 1), 5: (0, 2), 7: (0, 3), 48: (1, 0), 50: (1, 1), 52: (1, 2), 54: (1, 3)}


In [4]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

DATA_FILE  = 'Run2_list_with_trig_padd.dat'
CHUNK_SIZE = 1000    # events parsed per iteration — controls peak memory
HG_MIN     = 250     # minimum HG ADC value for a hit to count

# Main loop

In [ ]:
# ---------------------------------------------------------------------------
# Main loop — process file in chunks, apply cuts, accumulate rows
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------------------
PAIR_WINDOW_US      = 1.0   # max |Δt| µs to consider board records as the same physical event
MIN_PLANES_HIT      = 1     # min distinct planes (any board) with qualifying hits
MIN_TRIG_PLANES_HIT = 1     # min distinct Board-2 planes with qualifying hits (0 = disabled)
_board_maps         = {0: (board0_map, 0), 1: (board1_map, 1), 2: (board2_map, 2)}


# ---------------------------------------------------------------------------
# Helper: apply cuts to one grouped physical event and emit rows
# ---------------------------------------------------------------------------
def _process_physical_event(ev_list, phys_num, rows):
    """
    ev_list : 1–3 raw-event dicts from different boards (board_id 0, 1, and/or 2).
    Returns True if the event passes cuts and rows were added.
    """
    qualifying = []
    for ev in ev_list:
        ch_map, side = _board_maps.get(ev['board_id'], (board0_map, ev['board_id']))
        for h in ev['hits']:
            if h['energy_hg'] is not None and h['energy_hg'] > HG_MIN:
                layer, pos = ch_map.get(h['channel'], (None, None))
                qualifying.append(dict(
                    channel   = h['channel'],
                    side      = side,
                    layer     = layer,
                    position  = pos,
                    energy_lg = h['energy_lg'],
                    energy_hg = h['energy_hg'],
                    toa       = h['toa'],
                    tot       = h['tot'],
                ))

    # Cut: >= MIN_PLANES_HIT distinct layers (any board) with qualifying hits
    planes_hit = {h['layer'] for h in qualifying if h['layer'] is not None}
    if len(planes_hit) < MIN_PLANES_HIT:
        return False

    # Cut: >= MIN_TRIG_PLANES_HIT distinct Board-2 layers with qualifying hits
    if MIN_TRIG_PLANES_HIT > 0:
        trig_planes = {h['layer'] for h in qualifying
                       if h['layer'] is not None and h['side'] == 2}
        if len(trig_planes) < MIN_TRIG_PLANES_HIT:
            return False

    ts      = min(ev['timestamp_us'] for ev in ev_list)
    trig_id = ev_list[0]['trigger_id']

    for h in qualifying:
        rows.append(dict(
            event_number = phys_num,
            timestamp_us = ts,
            trigger_id   = trig_id,
            channel      = h['channel'],
            side         = h['side'],
            layer        = h['layer'],
            position     = h['position'],
            energy_lg    = h['energy_lg'],
            energy_hg    = h['energy_hg'],
            toa          = h['toa'],
            tot          = h['tot'],
        ))
    return True


# ---------------------------------------------------------------------------
# Main loop — chunk-read, group boards within PAIR_WINDOW_US, apply cuts
#
# Pairing strategy (supports board_id 0, 1, 2):
#   • pending dict holds the most-recent unmatched event per board.
#   • On each new event:
#       1. Flush any pending events too old to match (> PAIR_WINDOW_US away).
#       2. Flush any stale event from this same board.
#       3. Add current event to pending.
#       4. As soon as boards 0 AND 1 are both pending and within the time
#          window, commit the physical event — including board 2 if it is
#          also pending and within the window.
# ---------------------------------------------------------------------------
rows             = []
events_processed = 0
events_kept      = 0
chunks_done      = 0

pending = {}    # board_id -> most-recent unmatched raw event

for chunk in iter_events_chunked(DATA_FILE, chunk_size=CHUNK_SIZE):
    for ev in chunk:
        events_processed += 1
        board = ev['board_id']
        ts    = ev['timestamp_us']

        # Step 1 — flush pending events that are too old to pair with this one
        stale = [b for b, pev in list(pending.items())
                 if abs(pev['timestamp_us'] - ts) > PAIR_WINDOW_US]
        for b in stale:
            if _process_physical_event([pending.pop(b)], events_kept, rows):
                events_kept += 1

        # Step 2 — flush stale event from this same board
        if board in pending:
            if _process_physical_event([pending.pop(board)], events_kept, rows):
                events_kept += 1

        # Step 3 — park this event
        pending[board] = ev

        # Step 4 — commit when boards 0 and 1 are both ready and matched
        if 0 in pending and 1 in pending:
            ts0 = pending[0]['timestamp_us']
            ts1 = pending[1]['timestamp_us']
            if abs(ts0 - ts1) <= PAIR_WINDOW_US:
                group = [pending.pop(0), pending.pop(1)]
                # Include board 2 if it is also within the window
                if (2 in pending and
                        abs(pending[2]['timestamp_us'] - ts0) <= PAIR_WINDOW_US):
                    group.append(pending.pop(2))
                if _process_physical_event(group, events_kept, rows):
                    events_kept += 1

    chunks_done += 1
    if chunks_done % 100 == 0:
        print(f"  {events_processed:>10,} raw board events  |  "
              f"{events_kept:>8,} physical events kept  |  "
              f"{len(rows):>10,} hits")

# Flush any remaining unpaired events at EOF
for ev in pending.values():
    if _process_physical_event([ev], events_kept, rows):
        events_kept += 1

print(f"\nDone.")
print(f"  Raw board events processed : {events_processed:,}")
print(f"  Physical events kept       : {events_kept:,}")
print(f"  Total hits in dataframe    : {len(rows):,}")

In [ ]:
# ---------------------------------------------------------------------------
# Build DataFrame and set appropriate dtypes
# ---------------------------------------------------------------------------

df = pd.DataFrame(rows)

# Downcast integer columns where NaNs are not present.
# Fall back to nullable Int32 if the column contains NaN (e.g. unmapped channels
# where layer/position comes back as None).
for col in ('event_number', 'side', 'layer', 'position', 'channel'):
    try:
        df[col] = df[col].astype('int32')
    except (ValueError, TypeError) as e:
        print(f"Warning: '{col}' has NaN — using nullable Int32. ({e})")
        df[col] = df[col].astype('Int32')

# energy / timing columns may have NaN — use nullable Int32
for col in ('energy_lg', 'energy_hg', 'toa', 'tot'):
    df[col] = df[col].astype('Int32')
    
# Number of distinct layers hit per event
layers_per_event = df.groupby('event_number')['layer'].nunique().rename('n_layers')
df = df.join(layers_per_event, on='event_number')
df['n_layers'] = df['n_layers'].astype('int32')

print(df.dtypes)
print(f"\nDataFrame shape: {df.shape}")
df.head(5)

In [ ]:
# ---------------------------------------------------------------------------
# Quick sanity checks
# ---------------------------------------------------------------------------

print("=== HG ADC distribution (qualifying hits only) ===")
print(df['energy_hg'].describe())

print("\n=== Hits per layer ===")
print(df.groupby(['side', 'layer']).size().rename('hits'))

print("\n=== Hits per event (first 10 events) ===")
print(df.groupby('event_number').size().rename('n_hits').head(10))

import h5py
df.to_hdf('Run1027_events.h5', key='events', mode='w', complevel=5, complib='blosc')


In [ ]:
# ---------------------------------------------------------------------------
# Layer coverage analysis
# ---------------------------------------------------------------------------
ALL_LAYERS = {0, 1, 2, 3}

# One row per event: set of layers hit
layers_per_event = df.groupby('event_number')['layer'].apply(set)

n_total   = len(layers_per_event)
n_4layers = (layers_per_event.apply(len) == 4).sum()
print(f"Events with all 4 layers hit : {n_4layers:,} / {n_total:,}  "
      f"({100*n_4layers/n_total:.1f}%)")

# For events missing exactly one layer, identify which layer was skipped
# (MIN_PLANES_HIT=3 guarantees at least 3 layers, so at most one is missing)
incomplete = layers_per_event[layers_per_event.apply(len) < 4]
missed_layer = incomplete.apply(lambda s: next(iter(ALL_LAYERS - s)))

missed_counts    = missed_layer.value_counts().sort_index()
missed_fractions = missed_counts / n_total

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(missed_fractions.index, missed_fractions.values, width=0.6)
ax.set_xticks([0, 1, 2, 3])
ax.set_xlabel('Layer not hit')
ax.set_ylabel('Fraction of all events')
ax.set_title('Missed layer frequency (events with 3 of 4 layers hit)')
for i, v in missed_fractions.items():
    ax.text(i, v + 0.001, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# Matched side-0 / side-1 hits at the same (layer, position)
# ---------------------------------------------------------------------------

# Split by side, keep only the columns we need
s0 = (df[df['side'] == 0]
      [['event_number', 'layer', 'position', 'energy_hg', 'energy_lg', 'toa']]
      .copy())
s1 = (df[df['side'] == 1]
      [['event_number', 'layer', 'position', 'energy_hg', 'energy_lg', 'toa']]
      .copy())

# Inner join on (event, layer, position) → only cells read on both sides
matched = s0.merge(s1, on=['event_number', 'layer', 'position'],
                   suffixes=('_s0', '_s1'))

# Require both hits to have a valid TOA
mask    = matched['toa_s0'].notna() & matched['toa_s1'].notna()
matched = matched[mask]
print(f"Matched pairs with both TOA present: {len(matched):,}")

toa_sum = matched['toa_s0'].astype(float) + matched['toa_s1'].astype(float)

fig, axes = plt.subplots(3, 1, figsize=(15, 15))

# --- HG ADC ---
axes[0].hist(matched['energy_hg_s0'].astype(float), bins=100, alpha=0.6, label='Side 0')
axes[0].hist(matched['energy_hg_s1'].astype(float), bins=100, alpha=0.6, label='Side 1')
axes[0].set_xlabel('HG ADC (counts)')
axes[0].set_ylabel('Entries')
axes[0].set_xlim(100, 1000)
axes[0].set_yscale('log')
axes[0].set_title('HG ADC — matched pairs')
axes[0].grid(True, which='both', linestyle='--', alpha=0.5)
axes[0].legend()

# --- LG ADC ---
axes[1].hist(matched['energy_lg_s0'].astype(float), bins=100, alpha=0.6, label='Side 0')
axes[1].hist(matched['energy_lg_s1'].astype(float), bins=100, alpha=0.6, label='Side 1')
axes[1].set_xlabel('LG ADC (counts)')
axes[1].set_ylabel('Entries')
axes[1].set_title('LG ADC — matched pairs')
axes[1].grid(True, which='both', linestyle='--', alpha=0.5)
axes[1].legend()

# --- TOA sum ---
axes[2].hist(toa_sum, bins=100, color='steelblue')
axes[2].set_xlim(0, 100)  # zoom in on the main peak
axes[2].set_xlabel('TOA$_{s0}$ + TOA$_{s1}$ (LSB, 0.5 ns/LSB)')
axes[2].set_ylabel('Entries')
axes[2].grid(True, which='both', linestyle='--', alpha=0.5)
axes[2].set_title('Sum of TOA — matched pairs')


plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# Profile histogram: mean TOA vs LG ADC
# ---------------------------------------------------------------------------
from scipy.stats import binned_statistic

# Hits with both LG ADC and TOA present
mask = df['energy_lg'].notna() & df['toa'].notna()
x = df.loc[mask, 'energy_lg'].astype(float)
y = df.loc[mask, 'toa'].astype(float)

N_BINS    = 100
MIN_COUNT = 10   # suppress bins with too few entries

mean_toa, edges, _ = binned_statistic(x, y, statistic='mean',  bins=N_BINS)
std_toa,  _,    _ = binned_statistic(x, y, statistic='std',   bins=N_BINS)
count,    _,    _ = binned_statistic(x, y, statistic='count', bins=N_BINS)

centers = 0.5 * (edges[:-1] + edges[1:])
good    = count >= MIN_COUNT

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(centers[good], mean_toa[good],
            yerr=std_toa[good] / np.sqrt(count[good]),  # std error on the mean
            fmt='o', markersize=3, capsize=2, linewidth=1)
ax.set_xlabel('LG ADC (counts)')
ax.set_ylabel('Mean TOA (LSB,  0.5 ns / LSB)')
ax.set_title('Profile: TOA vs LG ADC')
ax.set_xscale('log')
ax.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Hits in profile: {mask.sum():,}  |  Bins plotted: {good.sum()} / {N_BINS}")


In [ ]:
# ---------------------------------------------------------------------------
# Time-walk fit (steeper):  TOA = C / (LG_ADC + a)^n
# ---------------------------------------------------------------------------
from scipy.optimize import curve_fit

def time_walk(x, C, a, n):
    x = np.asarray(x, dtype=float)
    result = C / (x + a)**n
    result[x > 1000] = C / (1000 + a)**n
    return result

# Restrict fit range to 100 < LG_ADC < 1500
fit_mask = (x_fit > 100) & (x_fit < 1500)
x_fit2   = x_fit[fit_mask]
y_fit2   = y_fit[fit_mask]
err_fit2 = err_fit[fit_mask]

popt, pcov = curve_fit(time_walk, x_fit2, y_fit2,
                       p0=[5000, 50, 1.0],
                       sigma=err_fit2,
                       absolute_sigma=True,
                       bounds=([0, 0, 0.1], [np.inf, 1000, 5.0]))

C_fit, a_fit, n_fit = popt
C_err, a_err, n_err = np.sqrt(np.diag(pcov))
print(f"Fit result:  C = {C_fit:.2f} ± {C_err:.2f}")
print(f"             a = {a_fit:.2f} ± {a_err:.2f}")
print(f"             n = {n_fit:.3f} ± {n_err:.3f}")

x_curve = np.linspace(x_fit.min(), x_fit.max(), 500)

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(x_fit, y_fit, yerr=err_fit,
            fmt='o', markersize=3, capsize=2, label='Profile data')
ax.plot(x_curve, time_walk(x_curve, *popt), 'r-', linewidth=2,
        label=f'Fit: C/(x+a)$^n$\n'
              f'C={C_fit:.1f}±{C_err:.1f}\n'
              f'a={a_fit:.1f}±{a_err:.1f}\n'
              f'n={n_fit:.3f}±{n_err:.3f}')
ax.axvspan(0, 100, alpha=0.1, color='gray', label='Excluded from fit')
ax.axvspan(1500, x_fit.max(), alpha=0.1, color='gray')
ax.set_xlabel('LG ADC (counts)')
ax.set_ylabel('Mean TOA (LSB,  0.5 ns / LSB)')
ax.set_title('Time-walk correction fit:  $C\,/\,(x+a)^n$')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# Mean absolute deviation of corrected TOA per event per side
# ---------------------------------------------------------------------------

# Step 1+2: subtract per-event-per-side mean, take abs, then average
mean_diff = (hits.groupby(['event_number', 'side'])['toa_corr']
               .apply(lambda x: (x - x.mean()).abs().mean())
               .unstack('side')
               .rename(columns={0: 'side0', 1: 'side1'})
               .dropna())

print(f"Events with both sides present: {len(mean_diff):,}")

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(mean_diff['side0'], mean_diff['side1'],
           s=1, alpha=0.5, linewidths=0)
ax.set_xlim(0,20)
ax.set_ylim(0,20)
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.set_xlabel('Mean |ΔTOA| — Side 0 (LSB)')
ax.set_ylabel('Mean |ΔTOA| — Side 1 (LSB)')
ax.set_title('Time-walk corrected TOA per event: Side 0 vs Side 1')
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# 2D density contours at 66% and 95%
# ---------------------------------------------------------------------------
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D

x = mean_diff['side0'].values
y = mean_diff['side1'].values

# KDE on a grid (may take a few seconds for large N)
kde = gaussian_kde(np.vstack([x, y]))

x_min, x_max = x.min(), x.max()
y_min, y_max = y.min(), y.max()
xx, yy = np.mgrid[x_min:x_max:200j, y_min:y_max:200j]
z = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)

# Find density thresholds that enclose 66% and 95% of the total probability
z_sorted = np.sort(z.ravel())[::-1]
cumsum   = np.cumsum(z_sorted) / np.sum(z_sorted)
level_66 = z_sorted[np.searchsorted(cumsum, 0.66)]
level_95 = z_sorted[np.searchsorted(cumsum, 0.95)]

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(x, y, s=1.5, alpha=0.8, linewidths=0, color='gray')
ax.contour(xx, yy, z, levels=sorted([level_95, level_66]),
           colors=['steelblue', 'tomato'], linewidths=1.5)

ax.legend(handles=[Line2D([0], [0], color='steelblue', lw=1.5, label='95%'),
                   Line2D([0], [0], color='tomato',    lw=1.5, label='66%')])
ax.set_xlabel('Mean |ΔTOA| — Side 0 (LSB)')
ax.set_ylabel('Mean |ΔTOA| — Side 1 (LSB)')
ax.set_title('Mean absolute corrected TOA deviation — density contours')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.grid(True)
plt.tight_layout()
plt.show()



In [ ]:
# ---------------------------------------------------------------------------
# Earliest (minimum) corrected TOA per event, per side, per layer
# ---------------------------------------------------------------------------

# For each (event, side, layer) group take the minimum corrected TOA
min_toa = (hits.groupby(['event_number', 'side', 'layer'])['toa_corr']
               .min()
               .reset_index())

fig, axes = plt.subplots(4, 2, figsize=(8, 14), sharex=True)

for layer in range(4):
    for side in range(2):
        ax   = axes[layer, side]
        data = min_toa[(min_toa['layer'] == layer) &
                       (min_toa['side']  == side)]['toa_corr']
        ax.hist(data, bins=101, range=(-50, 50))
        ax.set_title(f'Layer {layer}  —  Side {side}')
        ax.set_ylabel('Entries')
        if layer == 3:
            ax.set_xlabel('Min corrected TOA (LSB)')
        ax.grid(True)

ax.set_xlim(-50, 50)
plt.suptitle('Earliest corrected TOA per event', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# TOA(layer 3) - TOA(layer 0) per event per side
# ---------------------------------------------------------------------------

# Pivot so each row is (event, side) with columns for each layer's min TOA
layer_toa = (min_toa.pivot_table(index=['event_number', 'side'],
                                  columns='layer',
                                  values='toa_corr')
                     .dropna(subset=[0, 3]))   # require both layer 0 and 3

layer_toa['dt'] = layer_toa[3] - layer_toa[0]

# Split by side and join on event_number
dt_side0 = layer_toa.xs(0, level='side')['dt'].rename('side0')
dt_side1 = layer_toa.xs(1, level='side')['dt'].rename('side1')
dt = pd.merge(dt_side0.reset_index(),
              dt_side1.reset_index(),
              on='event_number').dropna()


print(f"Events with both sides and layers 0+3: {len(dt):,}")

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(dt['side0'], dt['side1'], s=1.5, alpha=0.5, linewidths=0)
ax.set_xlabel('TOA(layer 3) − TOA(layer 0)  —  Side 0 (LSB)')
ax.set_ylabel('TOA(layer 3) − TOA(layer 0)  —  Side 1 (LSB)')
ax.set_title('Layer transit time: Side 0 vs Side 1')
ax.set_xlim(-30, 30)
ax.set_ylim(-30, 30)
ax.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# Density contours: layer transit time side 0 vs side 1
# ---------------------------------------------------------------------------

x = dt['side0'].values
y = dt['side1'].values

# Restrict KDE to the plot range for speed
mask_range = (x > -30) & (x < 30) & (y > -30) & (y < 30)
kde = gaussian_kde(np.vstack([x[mask_range], y[mask_range]]))

xx, yy = np.mgrid[-30:30:200j, -30:30:200j]
z = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)

z_sorted = np.sort(z.ravel())[::-1]
cumsum   = np.cumsum(z_sorted) / np.sum(z_sorted)
level_66 = z_sorted[np.searchsorted(cumsum, 0.66)]
level_95 = z_sorted[np.searchsorted(cumsum, 0.95)]

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(x, y, s=2, alpha=0.8, linewidths=0, color='gray')
ax.contour(xx, yy, z, levels=sorted([level_95, level_66]),
           colors=['steelblue', 'tomato'], linewidths=1.5)
ax.legend(handles=[Line2D([0], [0], color='steelblue', lw=1.5, label='95%'),
                   Line2D([0], [0], color='tomato',    lw=1.5, label='66%')])
ax.set_xlim(-30, 30)
ax.set_ylim(-30, 30)
ax.set_xlabel('TOA(layer 3) − TOA(layer 0)  —  Side 0 (LSB)')
ax.set_ylabel('TOA(layer 3) − TOA(layer 0)  —  Side 1 (LSB)')
ax.set_title('Layer transit time: Side 0 vs Side 1')
ax.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# Linear fit to the core + Gaussian fit to residuals for the width
# ---------------------------------------------------------------------------
from scipy.stats import linregress
from scipy.optimize import curve_fit

# Use only points within the plot range for the fit
mask_core = ((dt['side0'] > -30) & (dt['side0'] < 30) &
             (dt['side1'] > -30) & (dt['side1'] < 30))
x_core = dt.loc[mask_core, 'side0'].values
y_core = dt.loc[mask_core, 'side1'].values

slope, intercept, r, p, se = linregress(x_core, y_core)
print(f"Linear fit:  y = {slope:.4f}·x + {intercept:.4f}   (R² = {r**2:.4f})")

# Residuals perpendicular to the line
residuals = y_core - (slope * x_core + intercept)

# Gaussian fit to the residuals
def gaussian(x, A, mu, sigma):
    return A * np.exp(-0.5 * ((x - mu) / sigma)**2)

counts, edges = np.histogram(residuals, bins=100, range=(-30, 30))
centers = 0.5 * (edges[:-1] + edges[1:])
popt, pcov = curve_fit(gaussian, centers, counts,
                       p0=[counts.max(), 0, 5])
A_fit, mu_fit, sigma_fit = popt
sigma_err = np.sqrt(np.diag(pcov))[2]
print(f"Residual width:  σ = {abs(sigma_fit):.3f} ± {sigma_err:.3f} LSB"
      f"  =  {abs(sigma_fit)*0.5:.3f} ± {sigma_err*0.5:.3f} ns")

# --- Scatter + fit line ---
x_line = np.array([-30, 30])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(dt['side0'], dt['side1'], s=1, alpha=0.15, linewidths=0, color='gray')
axes[0].plot(x_line, slope * x_line + intercept, 'r-', lw=2,
             label=f'y = {slope:.3f}x + {intercept:.3f}')
axes[0].set_xlim(-30, 30)
axes[0].set_ylim(-30, 30)
axes[0].set_xlabel('TOA(L3)−TOA(L0)  —  Side 0 (LSB)')
axes[0].set_ylabel('TOA(L3)−TOA(L0)  —  Side 1 (LSB)')
axes[0].set_title('Linear fit to core')
axes[0].legend()
axes[0].grid(True)

# --- Residuals with Gaussian fit ---
x_gauss = np.linspace(-30, 30, 500)
axes[1].bar(centers, counts, width=np.diff(edges), alpha=0.6, label='Residuals')
axes[1].plot(x_gauss, gaussian(x_gauss, *popt), 'r-', lw=2,
             label=f'σ = {abs(sigma_fit):.3f} LSB\n= {abs(sigma_fit)*0.5:.3f} ns')
axes[1].set_xlabel('Residual (LSB)')
axes[1].set_ylabel('Entries')
axes[1].set_title('Residuals from linear fit')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()
